# LLM01 Prompt Injection — Upload Artifacts & Run Evaluation

**OWASP Category**: LLM01 — Prompt Injection | **Risk Severity**: Critical

This notebook:
1. **Uploads** all artifact files (scenarios, checks, drivers) from the local folder structure and registers them in Okareo.
2. **Runs** the full LLM01 prompt injection test suite against your target AI agent.

Uploading is **idempotent** — re-running will not create duplicates (scenarios return existing, checks/drivers upsert).
Registered IDs are available in-memory for the evaluation steps below.

In [1]:
%pip install okareo python-dotenv --quiet


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import json
from pathlib import Path

# Add project root for owasp.common import
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))

from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver

from owasp.common import (
    init_okareo,
    parse_check_md,
    parse_driver_md,
    parse_check_py_meta,
    parse_check_py_metadata,
    parse_check_py,
    CodeCheckFromSource,
    build_target,
    SINGLE_TURN_DRIVER_TEMPLATE,
)

okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")
print(f"Category directory: {CATEGORY_DIR}")


✓ Okareo SDK initialized (key: ...a7sKA)
Category directory: /Users/guiair/dev/okareo/compliance-owasp/owasp/LLM01-prompt-injection


---
## Part 1 — Upload Artifacts

### Upload Scenarios

Scans `scenarios/` for `.jsonl` files and uploads each via `upload_scenario_set`.

In [3]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}  # name -> scenario object

for jsonl_path in sorted(scenarios_dir.glob("*.jsonl")):
    scenario_name = f"LLM01-{jsonl_path.stem}"
    print(f"Uploading scenario: {scenario_name} from {jsonl_path.name}")

    scenario = okareo.upload_scenario_set(
        scenario_name=scenario_name,
        file_path=str(jsonl_path),
    )
    registered_scenarios[scenario_name] = scenario
    print(f"  ✓ Registered: {scenario_name} (ID: {scenario.scenario_id})")

print(f"\nTotal scenarios uploaded: {len(registered_scenarios)}")

Uploading scenario: LLM01-crescendo-attack from crescendo-attack.jsonl
  ✓ Registered: LLM01-crescendo-attack (ID: 4b98f8b9-e50f-48a2-8846-483b81098178)
Uploading scenario: LLM01-direct-injection from direct-injection.jsonl
  ✓ Registered: LLM01-direct-injection (ID: 32aeac72-0a5d-47c1-a9e9-bfeaec7dc5bd)
Uploading scenario: LLM01-indirect-injection from indirect-injection.jsonl
  ✓ Registered: LLM01-indirect-injection (ID: 915c6054-2312-4f69-9e55-b2994a89f0b8)
Uploading scenario: LLM01-obfuscated-injection from obfuscated-injection.jsonl
  ✓ Registered: LLM01-obfuscated-injection (ID: c597037a-6868-40f3-9031-c932e5b8537b)
Uploading scenario: LLM01-payload-splitting from payload-splitting.jsonl
  ✓ Registered: LLM01-payload-splitting (ID: 1c91eddb-67e8-4011-aa40-9f30d2c6b882)

Total scenarios uploaded: 5


### Register Checks

Scans `checks/` for `.md` files, parses YAML front matter and prompt template,
and registers each via `create_or_update_check` using `ModelBasedCheck`.

In [4]:
checks_dir = CATEGORY_DIR / "checks"
registered_checks = {}  # name -> check id

for md_path in sorted(checks_dir.glob("*.md")):
    check_data = parse_check_md(md_path)
    print(f"Registering check: {check_data['name']} from {md_path.name}")

    check_obj = ModelBasedCheck(
        prompt_template=check_data["prompt_template"],
        check_type=CheckOutputType.PASS_FAIL,
    )
    result = okareo.create_or_update_check(
        name=check_data["name"],
        description=check_data["description"],
        check=check_obj,
    )
    registered_checks[check_data["name"]] = result.id
    print(f"  ✓ Registered: {check_data['name']} (ID: {result.id})")

print(f"\nTotal checks registered: {len(registered_checks)}")


Registering check: LLM01-compliance-drift-detector from compliance-drift-detector.md
  ✓ Registered: LLM01-compliance-drift-detector (ID: f19ee299-ef85-48b9-9706-5f20376f8f35)
Registering check: LLM01-injection-compliance-detector from injection-compliance-detector.md
  ✓ Registered: LLM01-injection-compliance-detector (ID: 30ad40c1-b32c-481c-997d-133e0e461030)

Total checks registered: 2


### Register Drivers

Scans `drivers/` for `.md` files, parses YAML front matter and persona prompt,
and registers each via `create_or_update_driver` using a `Driver` object.

In [5]:
drivers_dir = CATEGORY_DIR / "drivers"
registered_drivers = {}  # name -> driver object

for md_path in sorted(drivers_dir.glob("*.md")):
    driver_data = parse_driver_md(md_path)
    print(f"Registering driver: {driver_data['name']} from {md_path.name}")

    driver_obj = Driver(
        name=driver_data["name"],
        prompt_template=driver_data["prompt_template"],
        temperature=driver_data["temperature"],
    )
    result = okareo.create_or_update_driver(driver=driver_obj)
    registered_drivers[driver_data["name"]] = result
    print(f"  ✓ Registered: {driver_data['name']} (ID: {result.id})")

print(f"\nTotal drivers registered: {len(registered_drivers)}")


Registering driver: LLM01-jailbreak-escalator from jailbreak-escalator.md
  ✓ Registered: LLM01-jailbreak-escalator (ID: aacce38f-e5a7-4890-82e6-2b337fe63d75)

Total drivers registered: 1


### Artifact Upload Summary

In [6]:
print("=" * 60)
print("LLM01 Prompt Injection — Artifact Upload Summary")
print("=" * 60)
print(f"\nScenarios ({len(registered_scenarios)}):")
for name, sc in registered_scenarios.items():
    print(f"  • {name} → {sc.scenario_id}")
print(f"\nChecks ({len(registered_checks)}):")
for name, cid in registered_checks.items():
    print(f"  • {name} → {cid}")
print(f"\nDrivers ({len(registered_drivers)}):")
for name, drv in registered_drivers.items():
    print(f"  • {name} → {drv.id}")
print("\n✓ All artifacts ready. Proceeding to evaluation...")

LLM01 Prompt Injection — Artifact Upload Summary

Scenarios (5):
  • LLM01-crescendo-attack → 4b98f8b9-e50f-48a2-8846-483b81098178
  • LLM01-direct-injection → 32aeac72-0a5d-47c1-a9e9-bfeaec7dc5bd
  • LLM01-indirect-injection → 915c6054-2312-4f69-9e55-b2994a89f0b8
  • LLM01-obfuscated-injection → c597037a-6868-40f3-9031-c932e5b8537b
  • LLM01-payload-splitting → 1c91eddb-67e8-4011-aa40-9f30d2c6b882

Checks (2):
  • LLM01-compliance-drift-detector → f19ee299-ef85-48b9-9706-5f20376f8f35
  • LLM01-injection-compliance-detector → 30ad40c1-b32c-481c-997d-133e0e461030

Drivers (1):
  • LLM01-jailbreak-escalator → aacce38f-e5a7-4890-82e6-2b337fe63d75

✓ All artifacts ready. Proceeding to evaluation...


---
## Part 2 — Run Evaluation

### Configuration

The target agent is loaded from the shared `owasp/target.env` file (copy `owasp/target.env.example` and fill in your values).
All OWASP category notebooks reference this same file so that every control evaluates the same agent.

The target is registered as a `CustomEndpointTarget` with `TurnConfig` defining how to send messages.
Evaluations use `okareo.run_simulation()` — single-turn scenarios set `max_turns=1`; the multi-turn crescendo
simulation sets `max_turns=10`. See the [Okareo cookbook](https://colab.research.google.com/github/okareo-ai/okareo-cookbook/blob/main/notebooks/multiturn-evaluation/custom-endpoint-multiturn-demo.ipynb) for the reference pattern.

In [7]:
# Target loaded from owasp/target.env. To use a different config: target = build_target(CATEGORY_DIR, env_path="target.prod.env")
target = build_target(CATEGORY_DIR)
TARGET_NAME = target.name
print(f"✓ Target agent: {TARGET_NAME}")

SINGLE_TURN_CHECK = "LLM01-injection-compliance-detector"
MULTI_TURN_CHECK  = "LLM01-compliance-drift-detector"
DRIVER_NAME       = "LLM01-jailbreak-escalator"
MAX_TURNS         = 10

✓ Target agent: FinanceBot
  Max parallel requests: 10
  Endpoint: https://okareo-dev.ngrok.app/api/v1/financebot/next
  Response path: response.last_message


### Build Target

Constructs a `CustomEndpointTarget` from `owasp/target.env` using `TurnConfig` for the
next-turn endpoint and optional `SessionConfig` / `EndSessionConfig` for session management.

In [8]:
# Target built in config cell above via build_target(CATEGORY_DIR)


✓ Target built: FinanceBot


### Single-Turn Tests — Scenarios 1, 2, 4, 5

Each scenario runs via `okareo.run_simulation()` constrained to `max_turns=1`.
A pass-through driver delivers the scenario input verbatim to the target agent; the agent's
response is evaluated by the injection compliance detector check.

In [9]:
SINGLE_TURN_SCENARIOS = [
    "LLM01-direct-injection",
    "LLM01-indirect-injection",
    "LLM01-obfuscated-injection",
    "LLM01-payload-splitting",
]

single_turn_driver = Driver(
    temperature=0,
    name=f"{TARGET_NAME}-single-turn-driver",
    prompt_template=SINGLE_TURN_DRIVER_TEMPLATE,
)

single_turn_results = {}

for scenario_name in SINGLE_TURN_SCENARIOS:
    print(f"\n{'='*60}")
    print(f"Running: {scenario_name}")
    print(f"{'='*60}")
    try:
        scenario = registered_scenarios[scenario_name]

        test_run = okareo.run_simulation(
            target=target,
            driver=single_turn_driver,
            name=f"LLM01 Eval — {scenario_name}",
            api_key=OKAREO_API_KEY,
            first_turn="driver",
            scenario=scenario,
            max_turns=1,
            checks=[SINGLE_TURN_CHECK],
        )
        single_turn_results[scenario_name] = test_run
        print(f"  ✓ Test run complete: {test_run.id}")
        if hasattr(test_run, "app_link") and test_run.app_link:
            print(f"  View: {test_run.app_link}")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        single_turn_results[scenario_name] = None


Running: LLM01-direct-injection
  ✓ Test run complete: 04ffb08d-8bb9-45fd-be87-7d35b5dffc4b
  View: https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/04ffb08d-8bb9-45fd-be87-7d35b5dffc4b

Running: LLM01-indirect-injection
  ✓ Test run complete: aa38a67b-06ea-49ef-8461-6e071502c028
  View: https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/aa38a67b-06ea-49ef-8461-6e071502c028

Running: LLM01-obfuscated-injection
  ✓ Test run complete: ef95b11d-8bd7-4541-a708-1aebd015812a
  View: https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/ef95b11d-8bd7-4541-a708-1aebd015812a

Running: LLM01-payload-splitting
  ✓ Test run complete: 976cea67-fa7d-4fd9-9555-3f4f59e2e58f
  View: https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/976cea67-fa7d-4fd9-9555-3f4f59e2e58f


### Multi-Turn Simulation — Scenario 3: Crescendo Jailbreak

Runs a simulation using the jailbreak escalator driver and compliance drift detector check
via `okareo.run_simulation()` with `max_turns=10`.

In [10]:
MULTI_TURN_SCENARIO = "LLM01-crescendo-attack"

print(f"\n{'='*60}")
print(f"Running simulation: {MULTI_TURN_SCENARIO}")
print(f"Driver: {DRIVER_NAME} | Max turns: {MAX_TURNS}")
print(f"{'='*60}")

simulation_run = None
try:
    driver_reg = registered_drivers.get(DRIVER_NAME)
    if driver_reg is None:
        raise ValueError(f"Driver '{DRIVER_NAME}' not found in registered_drivers. Check upload step.")

    multi_turn_driver = Driver(
        temperature=driver_reg.temperature if hasattr(driver_reg, "temperature") else 0.6,
        name=DRIVER_NAME,
        prompt_template=driver_reg.prompt_template,
    )

    scenario = registered_scenarios[MULTI_TURN_SCENARIO]

    simulation_run = okareo.run_simulation(
        target=target,
        driver=multi_turn_driver,
        name=f"LLM01 Simulation — {MULTI_TURN_SCENARIO}",
        api_key=OKAREO_API_KEY,
        first_turn="target",
        scenario=scenario,
        max_turns=MAX_TURNS,
        checks=[MULTI_TURN_CHECK],
    )
    print(f"  ✓ Simulation complete: {simulation_run.id}")
    if hasattr(simulation_run, "app_link") and simulation_run.app_link:
        print(f"  View: {simulation_run.app_link}")
except Exception as e:
    print(f"  ✗ Error: {e}")


Running simulation: LLM01-crescendo-attack
Driver: LLM01-jailbreak-escalator | Max turns: 10
  ✓ Simulation complete: 571155e6-fad1-4fbe-8da5-d95f69f57c3a
  View: https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/571155e6-fad1-4fbe-8da5-d95f69f57c3a


### Results Summary

In [11]:
print("\n" + "=" * 60)
print("LLM01 PROMPT INJECTION — EVALUATION RESULTS")
print("OWASP Category: LLM01 | Risk Severity: Critical")
print("=" * 60)

all_results = dict(single_turn_results)
if simulation_run is not None:
    all_results[MULTI_TURN_SCENARIO] = simulation_run

print(f"\n{'Scenario':<42} {'Status':<10} {'Link / Run ID'}")
print("-" * 100)
for name, result in all_results.items():
    if result is None:
        print(f"{name:<42} {'ERROR':<10} N/A")
    else:
        link = getattr(result, "app_link", None) or result.id
        print(f"{name:<42} {'COMPLETE':<10} {link}")

errors = sum(1 for r in all_results.values() if r is None)
print(f"\nTotal evaluated: {len(all_results)} | Errors: {errors}")
if not errors:
    print("✓ All scenarios completed. See Okareo dashboard for full results.")


LLM01 PROMPT INJECTION — EVALUATION RESULTS
OWASP Category: LLM01 | Risk Severity: Critical

Scenario                                   Status     Link / Run ID
----------------------------------------------------------------------------------------------------
LLM01-direct-injection                     COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/04ffb08d-8bb9-45fd-be87-7d35b5dffc4b
LLM01-indirect-injection                   COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/aa38a67b-06ea-49ef-8461-6e071502c028
LLM01-obfuscated-injection                 COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/ef95b11d-8bd7-4541-a708-1aebd015812a
LLM01-payload-splitting                    COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/976cea67-fa7d-4fd9-9555-3f4f59e2e58f
LLM01-crescendo-attack                     COMPLETE   https://app.okareo.com/proj

### Detailed Results (Optional)

Retrieve per-row scores and model outputs for any completed test run.

In [12]:
# Uncomment to inspect a specific completed run in detail:
# from okareo_api_client.models import TestRunItem
# RUN_ID = "<paste_run_id_here>"
# detailed = okareo.get_test_run(RUN_ID)
# for row in (detailed.model_results or []):
#     print(f"Input:   {str(row.get('scenario_input', ''))[:80]}")
#     print(f"Output:  {str(row.get('model_output', ''))[:80]}")
#     print(f"Checks:  {row.get('check_scores', {})}")
#     print("-" * 40)